# OOP Project — NLPApp

**Topic:** 04.04 · **Level:** Intermediate · **Type:** PROJECT

## Welcome to your first OOP project

This session stops teaching syntax in isolation and puts it to work: you will build one complete, working application — **NLPApp** — using everything from the last three sessions. Classes, constructors, private attributes, menus, collections and API calls all come together in a single console program.

## What NLPApp does

NLPApp is a menu-driven command-line application that lets a user:

- **Register** a new account (name, email, password).
- **Log in** with an existing account.
- Once logged in, run real Natural Language Processing actions:
  - **NER** — named-entity recognition (extract names, places, languages from text).
  - **Language detection** — identify the language of a paragraph.
  - **Sentiment analysis** — classify text as joy, sadness, anger, fear, love or surprise.
- **Log out** and return to the first menu.

The NLP itself comes from the third-party **nlpcloud** API: we send text to a hosted model and parse the JSON it returns. If the API key or network fails during class, do not panic — the deliverable is understanding the *structure*: how a single class orchestrates a whole user journey.

## How it will be built

1. An OOP tour of the `nlpcloud` client and installing packages with `!pip install`.
2. Exploring the raw API: entity recognition, language detection, sentiment.
3. Building the `NLPApp` class with private state and menu methods.
4. Parsing the API response — pulling the highest-scoring label out of `scored_labels`.
5. A sprinkle of the `emoji` module for friendly output.

## Why a project matters

An application is different from a script: it keeps **state** across interactions, it hides data behind methods, and its code splits into small responsibilities. That is OOP in production. After this session you can genuinely say you have written a small but real software application.

**Prerequisite:** `!pip install nlpcloud` — install it before running the client cells.

# 1. The nlpcloud API

## 1.1 Connecting to a hosted model

`nlpcloud` wraps REST calls to hosted NLP models behind simple Python methods:

```python
import nlpcloud
client = nlpcloud.Client("finetuned-gpt-neox-20b", "YOUR_API_KEY")
```

- `Client(model_name, api_key)` returns an authenticated client object.
- The model name selects which model runs (`finetuned-gpt-neox-20b`, `distilbert-base-uncased-emotion`, `python-langdetect`).
- Calls cost quota and need the network — failures are usually network, quota or invalid key issues, not a bug in your code.
- Responses arrive as JSON (Python dicts) that you must parse.
- Create the client once and reuse it for every call.
- Keep the API key secret in production — never commit it to a repository.

The first cell connects and calls `client.entities(...)` for named-entity recognition, returning a dict whose `entities` value is a list of found entities with positions and types. The language of this API mirrors OOP nicely: the `client` object knows how to talk to a model, and calls like `client.entities(text, ...)` are simply methods on that object — the library is already object-oriented before any of your classes appear.

## Deep Dive : NLPApp — external API setup

```python
import nlpcloud
client = nlpcloud.Client("finetuned-gpt-neox-20b", "YOUR_API_KEY")
```

- **nlpcloud** — third-party Python library (an API for NLP models)
- `Client(model_name, api_key)` — authenticated client object
- Models: sentiment (`distilbert-base-uncased-emotion`), language detect (`python-langdetect`), text gen
- API key — your account secret (nothing works without it)
- Response: a JSON dict → must be parsed
- Calls cost money/quota — keep the key secret in production
- Structured as: statement object + method with text
- Create the client once, reuse it for many calls
- Errors: network, quota, auth — a try/except is required


In [ ]:
import nlpcloud

client = nlpcloud.Client("finetuned-gpt-neox-20b", "2b58d7fb9af09e617ee525e78c7766b6d8c5bb61", gpu=True, lang="en")
client.entities("""John Doe started learning Javascript when he was 15 years old. After a couple of years he switched to Python and starter learning low level programming. He is now a Go expert at Google.""", searched_entity="programming languages")

{'entities': [{'start': 26,
   'end': 36,
   'type': 'programming languages',
   'text': 'javascript'},
  {'start': 102,
   'end': 108,
   'type': 'programming languages',
   'text': 'python'},
  {'start': 165, 'end': 167, 'type': 'programming languages', 'text': 'go'}]}

# 2. Packages, Installation & API Tours

## 2.1 Installing with pip from the notebook

A Jupyter notebook lets you run shell commands by prefixing them with `!`:

```python
!pip install nlpcloud
```

- `!command` executes a shell command inside the notebook cell.
- `pip install packagename` downloads and installs a Python package.
- A package is a library of modules (`nlpcloud`, `emoji`).
- Install once — you do not reinstall on every run.
- Upgrades: `!pip install --upgrade nlpcloud`; check installed packages: `!pip list`.
- Internet is required during installation.

On Google Colab, a bare `pip install nlpcloud` line also works directly. After installing, the import + client cells connect to the API. The exploration cells fire real requests — entity recognition, sentiment analysis and language detection. Take the structure in even when responses vary with the network or key. Watch the order of operations: install first, import second, call third. If the import cell fails with `ModuleNotFoundError`, the installation did not finish — run the pip cell again and wait for `Successfully installed`.

## 2.2 Deep dive — a quick tour of three models

```python
client = nlpcloud.Client("distilbert-base-uncased-emotion", "KEY")
client.sentiment("...")     # scored_labels: joy, sadness, anger...

client = nlpcloud.Client("python-langdetect", "KEY")
client.langdetection("Et il parle aussi un peu français.")

client = nlpcloud.Client("finetuned-gpt-neox-20b", "KEY", gpu=True, lang="en")
client.entities(text, searched_entity="programming languages")
```

- `sentiment()` returns `scored_labels` — one entry per emotion with a probability score.
- `langdetection()` returns the detected language and its confidence.
- `entities()` returns identified entities inside the text — here it hunts for programming languages in a short developer bio.
- In the sample sentiment result, `joy` carries a score near `0.998` — the model is almost certain.

These raw API answers are the *data* your app will parse. Understanding the JSON shape is the key skill: the app's job is to read these dicts and turn them into simple, useful messages. Notice that sending the same paragraph to `sentiment()` twice yields nearly identical scores — the model is stable enough for a demo. What does vary is the size and order of the JSON, which is exactly why you must parse the answer instead of hard-coding it.

In [ ]:
pip install nlpcloud
# https://nlpcloud.com/home/playground/language-detection

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
import nlpcloud

client = nlpcloud.Client("distilbert-base-uncased-emotion", "2b58d7fb9af09e617ee525e78c7766b6d8c5bb61", gpu=False, lang="en")
client.sentiment("Look what's just come on the market in #ValThorens! A recently renovated, charming 6 bed duplex apartment in the heart of the resort with superb views!")

{'scored_labels': [{'label': 'sadness', 'score': 0.0001777907891664654},
  {'label': 'joy', 'score': 0.9987751841545105},
  {'label': 'love', 'score': 0.0004087708657607436},
  {'label': 'anger', 'score': 0.00012217740004416555},
  {'label': 'fear', 'score': 0.00011809932038886473},
  {'label': 'surprise', 'score': 0.0003979843750130385}]}

In [11]:
import nlpcloud

client = nlpcloud.Client("distilbert-base-uncased-emotion", "2b58d7fb9af09e617ee525e78c7766b6d8c5bb61", gpu=False, lang="en")
client.sentiment("Look what's just come on the market in #ValThorens! A recently renovated, charming 6 bed duplex apartment in the heart of the resort with superb views!")

{'scored_labels': [{'label': 'sadness', 'score': 0.0001777907891664654},
  {'label': 'joy', 'score': 0.9987751841545105},
  {'label': 'love', 'score': 0.00040877042920328677},
  {'label': 'anger', 'score': 0.00012217740004416555},
  {'label': 'fear', 'score': 0.00011809932038886473},
  {'label': 'surprise', 'score': 0.0003979839675594121}]}

In [4]:
import nlpcloud

client = nlpcloud.Client("python-langdetect", "3126efa8746a8c9a683e757205437143fa015ec5")
client.langdetection("""Et il parle aussi un peu français.""")

HTTPError: ignored

In [7]:
import nlpcloud

client = nlpcloud.Client("finetuned-gpt-neox-20b", "3126efa8746a8c9a683e757205437143fa015ec5", gpu=True, lang="en")
client.entities("""John Doe started learning Javascript when he was 15 years old. After a couple of years he switched to Python and starter learning low level programming. He is now a Go expert at Google.""", searched_entity="programming languages")

HTTPError: ignored

In [2]:
!pip install nlpcloud

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


# 3. The NLPApp Class

## 3.1 One class runs the whole app

The program is a single class whose constructor starts the menu:

```python
class NLPApp:
    def __init__(self):
        self.__database = {}    # private: user records
        self.__first_menu()     # the app starts immediately
```

- `__database` — a private dict mapping email → `[name, password]`. Encapsulation keeps user data inside the class.
- The constructor calls `__first_menu()` — creating the app IS launching the app.
- Each menu action is its own private method: `__register`, `__login`, `__sentiment_analysis`. Modularity = one responsibility per method.
- Methods call each other: after registering, control returns to `__first_menu()`; after a successful login, `__second_menu()` opens.

State persists across method calls because it lives on `self` — the user interacts with the same object that remembers the database. This is how real interactive applications are organized. Because every method is private, the outside world can only trigger the flow through the constructor — no stray calls can start the app in a broken state. That is encapsulation applied to behavior, not just to data.

## NLPApp class — OOP structure

```python
class NLPApp:
    def __init__(self):
        self.__database = {}    # private: users ka data
        self.__first_menu()     # constructor me hi menu dikhan

    def __first_menu(self): ...
    def __register(self, name, email, password): ...
    def run(self): ...
```

- The first menu is called from `__init__` → the app starts as soon as the object is created
- `__database` is private — encapsulation of user data
- Methods with the `__` prefix → controlled access
- Each menu option has its own method — modular code


## Deep Dive : NLPApp class — structure

```python
class NLPApp:
    def __init__(self):
        self.data = {}        # state
        self.client = None    # lazy init

    def login(self):
        ...
    def register(self):
        ...
    def start(self):
        # menu loop
```

- OOP structure of the project — a single class, methods: login/register/start
- __init__ — state (users dict, client)
- Methods group functionality: each API action is a method
- Menu-driven: a while loop + input
- CLI app pattern (console user interface)
- Methods do: sentiment / textgen / language
- State persists between method calls
- Design: one responsibility per method
- This is the pattern real projects use (MVC-lite)


In [2]:
import nlpcloud

class NLPApp:

  def __init__(self):

    self.__database = {}
    self.__first_menu()

  def __first_menu(self):

    first_input = input("""
    Hi! how would you like to proceed?
    1. Not a member? Register
    2. Already a member? Login
    3. Galti se aa gaye? Exit
    """)

    if first_input == '1':
      self.__register()
    elif first_input == '2':
      self.__login()
    else:
      exit()

  def __second_menu(self):

    second_input = input("""
    Hi! how would you like to proceed?
    1. NER
    2. Language Detection
    3. Sentiment Analysis
    4. Logout
    """)

    if second_input == '1':
      self.__ner()
    elif second_input == '2':
      self.__language_detection()
    elif second_input == '3':
      self.__sentiment_analysis()
    else:
      exit()

  def __register(self):
    name = input('enter name')
    email = input('enter email')
    password = input('enter password')

    if email in self.__database:
      print('email already exists')
    else:
      self.__database[email] = [name,password]
      print('registration successful.Now login')
      print(self.__database)
      self.__first_menu()


  def __login(self):
    
    email = input('enter email')
    password = input('enter password')

    if email in self.__database:
      if self.__database[email][1] == password:
        print('login successful')
        self.__second_menu()
      else:
        print('wrong password.Try again')
        self.__login()
    else:
      print('This email is not registered')
      self.__first_menu()

  def __sentiment_analysis(self):
    para = input('enter the paragraph')

    client = nlpcloud.Client("distilbert-base-uncased-emotion", "2b58d7fb9af09e617ee525e78c7766b6d8c5bb61", gpu=False, lang="en")
    response = client.sentiment(para)

    L = []
    for i in response['scored_labels']:
      L.append(i['score'])

    index = sorted(list(enumerate(L)),key=lambda x:x[1],reverse=True)[0][0]

    print(response['scored_labels'][index]['label'])
    self.__second_menu()


obj = NLPApp()


    Hi! how would you like to proceed?
    1. Not a member? Register
    2. Already a member? Login
    3. Galti se aa gaye? Exit
    1
enter namenitish
enter emailnitish@gmail.com
enter password1234
registration successful.Now login
{'nitish@gmail.com': ['nitish', '1234']}

    Hi! how would you like to proceed?
    1. Not a member? Register
    2. Already a member? Login
    3. Galti se aa gaye? Exit
    2
enter emailnitish@gmail.com
enter password1234
login successful

    Hi! how would you like to proceed?
    1. NER
    2. Language Detection
    3. Sentiment Analysis
    4. Logout
    3
enter the paragraphThis isn't just a beautifully crafted gangster film. Or an outstanding family portrait, for that matter. An amazing period piece. A character study. A lesson in filmmaking and an inspiration to generations of actors, directors, screenwriters and producers. For me, this is more: this is the definitive film. 10 stars out of 10.
joy

    Hi! how would you like to proceed?
    1. NE

# 4. Parsing the API Response

## 4.1 From raw scores to a human answer

The sentiment API hands back nested JSON; your app must extract the *most probable* label:

```python
d = {'scored_labels': [
    {'label': 'sadness', 'score': 0.98},
    {'label': 'joy',     'score': 0.01},
]}

L = [i['score'] for i in d['scored_labels']]
index = sorted(list(enumerate(L)), key=lambda x: x[1], reverse=True)[0][0]
d['scored_labels'][index]['label']   # 'sadness'
```

Breaking it down:

- `d['scored_labels']` → a list of dicts, each with `label` and `score`.
- `enumerate(L)` → pairs of `(position, score)` ready for sorting.
- `sorted(..., key=lambda x: x[1], reverse=True)` → highest score first.
- `[0][0]` → take the top pair, then its position index.
- That index finds the label with the maximum probability.

The whole trick is "extract scores → order them → pick the top label". The app performs exactly this inside `__sentiment_analysis`. Reading nested JSON is a transferable skill: every modern API returns dicts and lists, and the pattern stays the same — navigate one level at a time, collect what you need, and let the data shape drive your decisions.

## Parsing the API response

```python
d = {'scored_labels': [
    {'label': 'sadness', 'score': 0.98},
    {'label': 'joy', 'score': 0.001},
]}

L = [i['score'] for i in d['scored_labels']]
index = sorted(list(enumerate(L)), key=lambda x: x[1],
               reverse=True)[0][0]
d['scored_labels'][index]['label']
```

- The API's `scored_labels` list — a score for each label
- `enumerate` + `sorted(..., reverse=True)` — the index of the highest score
- `lambda x: x[1]` — sorts by score
- Top label = the emotion/language with the highest probability


## Deep Dive : Parsing the API response

```python
d = {'scored_labels': [{'label': 'sadness', 'score': 0.9809}, {'label': 'joy', 'score': 0.019}]}
L = []
for i in d['scored_labels']:
    L.append(i['score'])        # [0.98, 0.01]
index = L.index(max(L))         # highest probability
d['scored_labels'][index]['label']   # → 'sadness'
```

- The API returns a JSON dict — a nested structure
- `d['scored_labels']` → a list of score entries
- Each entry is a dict: `{'label': 'sadness', 'score': 0.98}`
- Extract the scores → find the max → get the label of the max
- `max(L)` + `L.index()` — the top prediction
- Alternative: `max(d['scored_labels'], key=lambda x: x['score'])`
- The data shape matters — understand the API docs structurally
- JSON→dict is directly usable (the json module parses it)
- Parsing pattern: extract → aggregate → decide


In [2]:
d = {'scored_labels': [{'label': 'sadness', 'score': 0.98093181848526}, {'label': 'joy', 'score': 0.001407247269526124}, {'label': 'love', 'score': 0.0004157320945523679}, {'label': 'anger', 'score': 0.01649504341185093}, {'label': 'fear', 'score': 0.00039679379551671445}, {'label': 'surprise', 'score': 0.00035347335506230593}]}

In [3]:
d

{'scored_labels': [{'label': 'sadness', 'score': 0.98093181848526},
  {'label': 'joy', 'score': 0.001407247269526124},
  {'label': 'love', 'score': 0.0004157320945523679},
  {'label': 'anger', 'score': 0.01649504341185093},
  {'label': 'fear', 'score': 0.00039679379551671445},
  {'label': 'surprise', 'score': 0.00035347335506230593}]}

In [33]:
L = []
for i in d['scored_labels']:
  L.append(i['score'])

index = sorted(list(enumerate(L)),key=lambda x:x[1],reverse=True)[0][0]

In [38]:
d['scored_labels'][index]['label']

'sadness'

# 5. The emoji Module

## 5.1 A friendly touch of fun

```python
import emoji
print(emoji.emojize('Python is :grinning_face:'))   # Python is 😀
```

- `emoji` is a small ecosystem library that replaces `:code:` placeholders with real emoji glyphs.
- Install it first: `!pip install emoji`.
- It is optional polish — nicer-looking output for a console app.
- The terminal must support the glyphs for the emoji to render.

The last cells install `emoji`, then convert `:grinning_face:` into a grinning face, proving the happy path works before you wire it into NLPApp. Emojis are the smallest of the three integrations here: like `nlpcloud`, they are a pip-installed friend you barely notice once installed — good practice for the install → import → use rhythm that every third-party library follows. That rhythm — install once, import in every notebook or module, then call — governs all three libraries in this project, and it is the single most transferable habit from it. When an import fails, the fix is reinstall or restart the kernel, not rewriting code.

## emoji module

```python
import emoji
print(emoji.emojize('Python is :grinning_face:'))
```

- `emoji.emojize()` — turns codes like `:grinning_face:` into emojis
- For showing output in the UI — a bit of fun
- You must install it first with `!pip install emoji`
- Showing an emoji in the NLP app's result = a user-friendly touch


## Deep Dive : emoji module

```python
import emoji
print(emoji.emojize('Python is :grinning_face:'))   # Python is
```

- emoji — a library that turns `:name:` text into emojis
- `emojize('text with :code:')` — replaces the placeholder
- Needs install: `!pip install emoji`
- Fun/UX — a friendly touch in the output
- Not core Python — an ecosystem package
- Alternative: unicode emoji directly in the string
- Printing emoji in the console needs font support
- Real use: notifications, fun features
- Which codes: `emojize(':name:', language='alias')`


In [9]:
import emoji
print(emoji.emojize('Python is :grinning_face:'))

Python is 😀


In [6]:
!pip install emoji

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 240 kB 15.2 MB/s 
  Created wheel for emoji: filename=emoji-2.2.0-py3-none-any.whl size=234926 sha256=41b6c4bb33328577a544347fe1c0258b19e2bac46989d0bb233c97ed1d66de05
  Stored in directory: /root/.cache/pip/wheels/f3/e3/f2/1de1c2e3ed742e1df73e0f15d58864e50c7e64f607b548d6cf
Successfully built emoji


## Summary

This project tied the whole OOP course together. You connected to a hosted NLP API with `nlpcloud.Client(model, key)`, discovered the JSON shapes returned by sentiment, language detection and entity recognition, and installed packages from inside the notebook with `!pip install`. Then you built `NLPApp`, a single class holding a private `__database` of users, whose constructor immediately launches a menu-driven loop: register, login, run NER / language detection / sentiment, and log out. You parsed the API's `scored_labels` by extracting scores, sorting with `enumerate` and a lambda, and picking the top label — and learned that an application is really objects holding state and methods orchestrating a user journey. The emoji module added a friendly finish. Whether the API key works live or not, you now know how to structure a real, stateful, interactive Python application with OOP — the blueprint every real project follows.